# 이전 4시간 구성 · 선택 복습
현재 필수 수업은 [교재 안내](../../README.md)를 따릅니다. 이 파일은 이전 상세 실습 보존본입니다.


# LangChain으로 이미지에 질문하기

**목표:** 텍스트와 이미지가 함께 들어가는 메시지를 만들고 원본과 답을 비교합니다.

60분 수업: 그림·개념 설명 → 실행 전 예측 → 코드 실행 → 원문·기록 비교. 앞 세 교시는 휴식 10분 포함, 마지막 교시는 종합 평가 포함입니다.

[교재 안내](../../README.md) · [슬라이드 62~71쪽](https://docs.google.com/presentation/d/1IIy5LG5Yue5sJR5DOzmtW6jOuX0DDIZb3LsswTamjcw/edit) · [배포 PDF](../../../슬라이드/강의자료_2일차_v2.pdf#page=62)

**커널:** Python (Day2 LangChain). 설치는 개인 컨테이너 Terminal에서 `bash /workspace/04.LLM_v2/scripts/install_langchain.sh`를 실행합니다. 설치가 끝나면 커널을 선택하고 아래 환경 셀부터 실행하세요. 본문 코드는 실행 가능한 기준 구현이며 마지막에는 별도의 TODO가 있습니다. 코드 작성 여부와 수정할 파일을 TODO 안내에서 확인하세요. TODO는 수업 마지막 10~15분 동안 진행합니다.

In [ ]:
from pathlib import Path
import json, os, sys
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents) if (p/'04.LLM_v2').is_dir()),None)
if ROOT is None:raise RuntimeError('배포 폴더 안에서 노트북을 여세요.')
COURSE=ROOT/'05.VLM_v2'
sys.path.insert(0,str(COURSE))
from vlm_v2.core import *
print('Python:',sys.executable)
print('교재:',COURSE)

## 1. 오전의 메시지에 이미지를 추가합니다

LLM은 글, VLM은 이미지와 글을 함께 입력받습니다.

LangChain의 HumanMessage에 두 콘텐츠를 넣습니다.

이미지 계산은 공용 Ollama 모델이 수행합니다.

관찰 결과는 입력 이미지와 대조합니다.

오전의 invoke와 AIMessage 사용법을 재사용합니다. VLM이라는 이름이 모든 모델의 이미지 지원을 보장하지는 않습니다. 본 배포의 gemma4:e2b-it-qat를 사용하고 gemma3:1b는 텍스트 요약에 사용합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

![오전의 메시지에 이미지를 추가합니다](../../assets/01_vlm_basics-01.svg)

## 2. 설치 환경과 모델을 확인합니다

오전의 Day2 LangChain 커널을 그대로 사용합니다.

실습 컨테이너에 설치된 OpenCV와 Pillow를 사용합니다.

VLM 모델의 vision capability를 확인합니다.

모델을 새로 다운로드하는 절차는 없습니다.

개인 컨테이너에서 check_langchain.py --server를 실행하면 두 모델과 이미지 지원을 확인합니다. LangChain 패키지의 이미지 입력 지원과 모델 태그의 이미지 지원은 각각 확인해야 합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

**개인 컨테이너 Terminal 명령** (Ollama 접속 명령은 Jetson 호스트):

```bash
cd /workspace
PY=Data/venvs/day2-langchain/bin/python
"$PY" 04.LLM_v2/scripts/check_langchain.py \
  --server
```

## 3. 픽셀과 해상도는 입력의 정보량입니다

이미지는 가로·세로·색 채널의 배열입니다.

해상도가 낮으면 작은 물체·글자를 잃을 수 있습니다.

크게 보낸다고 항상 정확해지지는 않습니다.

파일 크기·처리 시간도 함께 봅니다.

이미지의 배열 크기와 파일의 압축 크기는 다릅니다. JPEG는 압축된 파일이고 모델은 디코딩된 시각 정보를 처리합니다. 원본을 먼저 보고 질문이 이미지에서 답할 수 있는 것인지 확인하세요.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
from PIL import Image
from IPython.display import display
image_path=ROOT/'Data/images/05.VLM/road.png'
with Image.open(image_path) as im:
    print(im.size,im.mode)
    display(im.copy())

## 4. Base64는 전송 표현입니다

파일 바이트를 문자로 표현합니다.

이미지 의미를 추출한 임베딩이 아닙니다.

data URI에는 MIME과 Base64를 넣습니다.

큰 문자열 전체 대신 길이와 구조를 확인합니다.

Base64는 원본 파일보다 문자열이 길어집니다. 인코딩 자체가 물체를 인식하지 않습니다. LangChain은 image_url 콘텐츠를 Ollama 요청의 images 형태로 변환합니다. 로컬 파일 경로 문자열만 전송하면 서버가 파일을 읽는 것은 아닙니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

![Base64는 전송 표현입니다](../../assets/01_vlm_basics-04.svg)

## 5. 이미지 메시지의 구조를 읽습니다

HumanMessage.content가 콘텐츠 목록입니다.

text 항목은 질문입니다.

image_url 항목은 인코딩한 이미지입니다.

파일 형식과 MIME이 일치해야 합니다.

image_message는 Pillow로 실제 파일 형식을 확인하고 MIME을 선택합니다. 확장자만 바꾼 파일은 검증에서 다르게 드러날 수 있습니다. 메시지를 print하면 큰 Base64가 출력되므로 타입과 길이를 선택해서 봅니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
message=image_message(image_path,'이 장면에서 보이는 것을 두 문장으로 설명하세요.')
print(message.type)
print(message.content[0])
print(message.content[1]['type'],len(message.content[1]['image_url']))

## 6. 실제 VLM 질의를 보내고 기록합니다

make_model(vision=True)로 VLM을 선택합니다.

observe는 요청·응답·시간·원본 해시를 묶습니다.

첫 실습은 자유로운 장면 설명입니다.

응답을 보기 전에 기대 관찰을 적습니다.

이 교재는 모델의 긴 reasoning 출력 대신 이미지 설명에 출력 예산을 사용하도록 reasoning=False를 설정합니다. 실제 생성은 비결정적일 수 있고 공유 서버 부하로 지연이 달라집니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
vlm=make_model(vision=True,num_predict=384)
result=observe(vlm,image_path,structured=False,question='이미지에서 직접 보이는 대상을 두 문장으로 설명하세요.')
run=new_run('image');append_record(run/'observations.jsonl',result)
print(result['status'],result['raw'],result['error'])
print('저장:',run)

## 7. 관찰과 추측은 다른 종류의 문장입니다

색·형태·보이는 물체는 원본에서 확인합니다.

가려진 물체의 존재를 단정하지 않습니다.

한 장으로 이동 방향이나 원인을 확정하지 않습니다.

보이지 않는 부분은 불확실성으로 기록합니다.

도로 사진에 차량이 보인다는 것과 차량이 움직이고 있다는 것은 다릅니다. 시간 정보가 없는 정지 이미지에서 실제 속도나 전후 변화를 주장하면 근거가 부족합니다. 다음 교시의 프롬프트에도 이 규칙을 반영합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

## 8. 입력 해시와 시각은 추적의 기준입니다

SHA-256으로 같은 파일인지 대조합니다.

모델 이름과 질문을 기록합니다.

요청 시간과 촬영 시각은 다른 값입니다.

오래된 프레임에 대한 답을 현재 상황으로 혼동하지 않습니다.

해시는 이미지 의미의 유사도를 뜻하지 않습니다. 바이트가 바뀌면 동일 장면이라도 값이 달라질 수 있습니다. 원본을 잃으면 응답을 다시 검토하기 어려우므로 카메라 실습에서는 선택 프레임을 함께 저장합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

![입력 해시와 시각은 추적의 기준입니다](../../assets/01_vlm_basics-08.svg)

## 9. 오류를 보고 원인을 나눕니다

연결 오류와 모델 미준비를 구분합니다.

이미지 파일·형식·모델 capability를 확인합니다.

출력 상한으로 잘린 응답도 기록합니다.

오류 결과를 관찰 사실로 쓰지 않습니다.

파일 입력 실패는 모델 추론 전에 발생합니다. 요청 오류는 서버·통신·시간 상한 등의 문제이고 형식 오류는 응답 구조가 틀린 경우입니다. 의미 정답은 원본을 보는 사람이 별도로 평가합니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

In [ ]:
print({k:result.get(k) for k in ['id','status','input_sha256','seconds','error']})
print(result.get('metadata',{}))

## 10. 1교시 정리 · 이미지부터 응답까지 설명합니다

text와 image가 어디에 들어가는지 설명합니다.

Base64와 임베딩을 구분합니다.

원본·답변·시간·해시 기록을 확인합니다.

추가 질문은 같은 이미지로 비교합니다.

이론 25분, 실습 20분, 정리 5분, 휴식 10분입니다. 실습을 빨리 끝냈다면 사진에 없는 상황을 질문하고 어디까지 답할 수 있는지 판단하세요. 근거 없는 단정을 찾는 것이 목표입니다.

**설명해 보기:** 입력·중간값·출력 중 이 단계에서 바뀌는 것은 무엇인가요?

## 마지막 TODO — 텍스트와 이미지 콘텐츠 구성

**코드 작성: 있음 · 10분 · 수업 시간 안에 진행**

수정 파일: [practice/image_content.py](../../practice/image_content.py). JupyterLab 편집기에서 함수를 완성하고 저장한 뒤 아래 로딩 셀과 검사 셀을 다시 실행하세요. 노트북은 학생 파일을 덮어쓰지 않습니다.

**입력·출력 계약:** question과 준비된 data_uri를 받아 LangChain의 콘텐츠 목록을 만드세요. 첫 항목은 {"type":"text","text":question.strip()}, 둘째 항목은 {"type":"image_url","image_url":data_uri}입니다. question은 비어 있지 않은 문자열이어야 하며 data_uri는 data:image/로 시작하고 ;base64, 뒤 본문이 있어야 합니다. 잘못된 입력은 ValueError입니다. 실제 이미지 형식 검증은 앞 단계에서 완료합니다.

**힌트:** 문자열 strip/startswith와 dict 두 개를 담는 list를 사용하세요. Base64 전체를 출력하지 마세요.

미완성이면 마지막 검사에서 NotImplementedError가 나는 것이 정상입니다. 검사 통과 후 아래 적용 예제의 중간값을 설명하세요. 이 예제는 학생 함수를 직접 사용하며 본문의 기준 클래스/카메라 프로그램을 자동 교체하지 않습니다.

In [ ]:
import importlib.util, inspect, os
runner_path=COURSE/'practice/runner.py'
spec=importlib.util.spec_from_file_location('day2_todo_runner',runner_path)
runner=importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
VARIANT=os.environ.get('DAY2_TODO_VARIANT','exercise')
task=runner.load('image_content',VARIANT)
print('선택 구현:',VARIANT)
print(inspect.getsource(task.build_image_content))

In [ ]:
print(runner.check('image_content',task))

from langchain_core.messages import HumanMessage
from vlm_v2.core import image_message
path=ROOT/'Data/images/05.VLM/road.png'
# 실제 파일 형식 확인·Base64 인코딩은 앞 교시의 부품을 재사용합니다.
encoded=image_message(path,'인코딩 준비').content[1]['image_url']
message=HumanMessage(content=task.build_image_content('  직접 보이는 것은?  ',encoded))
print(message.content[0])
print(message.content[1]['type'],len(message.content[1]['image_url']))

### TODO 완료와 참고 정답

- [ ] 함수를 직접 작성하고 검사 셀을 통과했습니다.
- [ ] 경계 입력 한 개를 추가해 예상 결과를 설명했습니다.
- [ ] 적용 예제의 입력과 반환값을 설명했습니다.

강사가 정한 직접 시도 시간 뒤 [참고 정답](../../practice/solutions/image_content.py)을 비교하세요. 참고 정답을 실행했다고 학생 파일이 완성되는 것은 아닙니다.

**해설:** 이미지 전송과 질문을 하나의 HumanMessage로 묶습니다. 이 함수는 메시지 구조를 만들며 물체를 인식하거나 Base64가 진짜 이미지인지 판정하는 함수가 아닙니다.

추가 모델 요청 없이 함수와 데이터 흐름을 검사합니다. 실제 모델·카메라 결과 평가는 본문 실습에서 수행합니다.

## 완료 확인

실제 입력과 출력 기록을 확인하고, 형식·근거·지연을 구분해 설명합니다. 오류는 숨기지 않고 원문과 함께 남깁니다. 추가 실험은 한 조건만 바꿔 비교합니다.